# GRPO Qwen hasil SFT sendiri · Kendrick Filbert

Dataset tetap Alpaca Indonesia yang ditentukan. **Draf CPU: tidak ada training GRPO yang diklaim.**

Jika memakai GPU baru, jalankan `pip install -r requirements-training.txt` dan
`pip install -e . --no-deps`, lalu `pip check` dan restart kernel. Rincian tersedia di
`docs/RUNBOOK_GPU.md`. Ambil token melalui environment/Colab Secrets, jangan tempel di sel.
Untuk mengaktifkan training, set `LEGAL_RAG_TRAIN=1` **sebelum Run All**.
Output CPU yang melewati sel GPU tidak memenuhi persyaratan eksekusi training submission.

In [ ]:
import os, sys, json, subprocess
from pathlib import Path
ROOT = Path(os.environ.get("LEGAL_RAG_ROOT", Path.cwd())).resolve()
assert (ROOT / "src/legal_rag").is_dir(), "Set LEGAL_RAG_ROOT to the extracted project folder"
sys.path.insert(0, str(ROOT / "src"))
os.environ["PYTHONPATH"] = str(ROOT / "src")
os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_FLAX", "0")
from legal_rag.config import settings, read_json, write_json
cfg = settings()
RUN_TRAINING = os.getenv("LEGAL_RAG_TRAIN") == "1"
if RUN_TRAINING:
    import unsloth  # Must precede transformers / sentence_transformers in this kernel.
print("Nama: Kendrick Filbert")
print("Mode:", "GPU training enabled" if RUN_TRAINING else "CPU verification; GPU training NOT executed")
def cli(*args):
    subprocess.run([sys.executable, "-m", "legal_rag.cli", *args], cwd=ROOT, check=True)


## Reward shaping dan ringkasan alasan
Tag `<think>` berisi ringkasan alasan singkat yang dapat diperiksa, kemudian jawaban akhir. Reward panjang/format tidak menjamin kebenaran. Correctness mengukur ground truth pada jawaban akhir saja; ringkasan sebelum final tidak dapat memperoleh skor correctness.

In [ ]:
import inspect
from legal_rag.rewards import format_reward_func, reasoning_length_reward, correctness_reward, language_reward_func
for fn in [format_reward_func, reasoning_length_reward, correctness_reward, language_reward_func]:
    print(inspect.getsource(fn))

In [ ]:
# Input berikut adalah fixture unit test, BUKAN keluaran model.
fixtures = ["<think>x", "<think>"+"a"*50+"</think>Jawab", "<think>"+"a"*200+"</think>Jawab", "<think>x</think><think>y</think>Jawab"]
print("format fixture:", format_reward_func(fixtures))
print("length fixture:", reasoning_length_reward(fixtures))
assert reasoning_length_reward(["<think>"+"a"*200]) == [1.0]
assert format_reward_func(["<think>x</think><think>y</think>Jawab"]) == [0.5]
assert correctness_reward(["<think>hak lembur</think>tidak tahu"], ["hak lembur"]) == [0.0]
print("Reward contract checks passed")

## Memuat ulang merged SFT mahasiswa
GRPO menolak berjalan jika receipt publik SFT sendiri tidak tersedia. Native template yang memprefill `<think>` dihapus suffix-nya saat membuat prompt GRPO agar tag harus dihasilkan model dan dapat dinilai reward.

In [ ]:
from legal_rag.hub import verified_repo
try:
    repo, revision = verified_repo("sft")
    print("Verified student SFT:", repo, revision)
except RuntimeError as exc:
    print(str(exc))
print(json.dumps(cfg["grpo"], indent=2))

## GRPOTrainer
Konfigurasi awal: 200 steps, num_generations=2, completion 384, akumulasi 4, beta 0.04, non-vLLM. TRL 1.15 membatasi prompt melalui filter token karena max_prompt_length sudah dihapus dari GRPOConfig. Efektif batch harus habis dibagi jumlah generasi.

In [ ]:
if RUN_TRAINING:
    cli("train", "grpo")
else:
    print("NOT RUN: tidak ada GPU/runtime dan model SFT publik mahasiswa.")

In [ ]:
path = ROOT / "artifacts/reports/grpo.json"
if path.exists():
    report = read_json(path)
    print(json.dumps({k:v for k,v in report.items() if k != "log_history"}, indent=2))
    print("Last real logs:", report["log_history"][-5:])
else:
    print("Belum ada log/reward training GRPO; tidak ada klaim peningkatan kualitas.")

In [ ]:
if RUN_TRAINING:
    cli("publish", "grpo")
else:
    print("NOT PUBLISHED: belum ada model GRPO yang telah dilatih.")

Lanjutkan notebook RAG untuk menguji model GRPO dengan pertanyaan wajib staf admin lembur 3 jam. Jangan mengganti output model dengan contoh rubrik. Tinjau nilai reward, clipping, keragaman generasi, dan kualitas final; reward tinggi dapat terjadi tanpa peningkatan akurasi.